In [1]:
# Uninstall and reinstall transformers with specific version
!pip uninstall transformers -y
!pip install transformers==4.36.0

Found existing installation: transformers 4.36.0
Uninstalling transformers-4.36.0:
  Successfully uninstalled transformers-4.36.0
  Using cached transformers-4.36.0-py3-none-any.whl.metadata (126 kB)
Using cached transformers-4.36.0-py3-none-any.whl (8.2 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.0 which is incompatible.


In [4]:
import torch
import transformers
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import re

print(f"Torch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")

Torch version: 2.9.1+cu128
Transformers version: 4.36.0


In [5]:
# Load GPT-2 model and tokenizer using explicit class names
model_name = "gpt2"  # You can also use "gpt2-medium"

print("Loading model and tokenizer...")

try:
    # Method 1: Using specific GPT2 classes
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    model = GPT2LMHeadModel.from_pretrained(model_name)

    # Set padding token
    tokenizer.pad_token = tokenizer.eos_token

    print("✓ Model and tokenizer loaded successfully!")
    print(f"Model: {model_name}")
    print(f"Number of parameters: {model.num_parameters():,}")

except Exception as e:
    print(f"Error loading model: {e}")
    print("\nTrying alternative method...")

    # Method 2: Try AutoModel with pipeline
    from transformers import pipeline
    generator = pipeline('text-generation', model='gpt2')
    print("✓ Loaded using pipeline!")

Loading model and tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✓ Model and tokenizer loaded successfully!
Model: gpt2
Number of parameters: 124,439,808


In [6]:
# Step 1: Import all necessary libraries
import torch
import transformers
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import re

print(f"Torch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print()

# Step 2: Load model and tokenizer
print("Loading GPT-2 model and tokenizer...")
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Set padding token
tokenizer.pad_token = tokenizer.eos_token

print("✓ Model and tokenizer loaded successfully!")
print(f"Model: {model_name}")
print(f"Number of parameters: {model.num_parameters():,}")
print()

# Step 3: Implement filtering mechanism
def is_python_coding_question(prompt):
    """
    Check if the prompt is related to Python coding.
    Returns True if it's a coding question, False otherwise.
    """
    prompt_lower = prompt.lower()

    # Python coding keywords
    coding_keywords = [
        'python', 'code', 'function', 'class', 'def', 'import',
        'variable', 'list', 'dictionary', 'loop', 'for', 'while',
        'if', 'else', 'elif', 'return', 'print', 'syntax',
        'error', 'debug', 'program', 'script', 'module',
        'package', 'pip', 'numpy', 'pandas', 'algorithm',
        'data structure', 'array', 'string', 'integer',
        'exception', 'try', 'except', 'lambda', 'decorator',
        'generator', 'iterator', 'comprehension', 'tuple',
        'set', 'method', 'attribute', 'inheritance', 'oop',
        'iterate', 'compile', 'runtime'
    ]

    # Check keywords
    for keyword in coding_keywords:
        if keyword in prompt_lower:
            return True

    # Check patterns
    coding_patterns = [
        r'how to .* in python',
        r'write .* code',
        r'create .* function',
        r'implement .*',
        r'\.py\b',
        r'def\s+\w+\(',
        r'import\s+\w+',
    ]

    for pattern in coding_patterns:
        if re.search(pattern, prompt_lower):
            return True

    return False

# Step 4: Generate response function
def generate_response(prompt, max_length=150, temperature=0.7):
    """
    Generate a response using the GPT-2 model.
    """
    # Encode input
    inputs = tokenizer.encode(prompt, return_tensors="pt")

    # Create attention mask
    attention_mask = torch.ones(inputs.shape, dtype=torch.long)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            attention_mask=attention_mask,
            max_length=max_length,
            temperature=temperature,
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            no_repeat_ngram_size=2
        )

    # Decode response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Step 5: Main inference function with filtering
def code_focused_inference(user_prompt):
    """
    Main function that filters prompts and generates responses
    only for Python coding questions.
    """
    print(f"\n{'='*70}")
    print(f"User Prompt: {user_prompt}")
    print('='*70)

    # Check if Python coding question
    if is_python_coding_question(user_prompt):
        print("✓ This is a Python coding question. Generating response...\n")

        # Enhance prompt for better responses
        enhanced_prompt = f"Python programming question: {user_prompt}\n\nAnswer:"

        response = generate_response(enhanced_prompt, max_length=200)

        print(f"Response:\n{response}")
        return response
    else:
        print("✗ This is NOT a Python coding question.\n")

        predefined_message = ("I'm sorry, but I can only answer questions related to "
                            "Python coding. Please ask me a Python programming question!")

        print(f"Response:\n{predefined_message}")
        return predefined_message

# Step 6: Test the implementation
print("\n" + "="*70)
print("TESTING CODE-FOCUSED INFERENCE SYSTEM")
print("="*70)

# Test Case 1: Python coding question
code_focused_inference("How do I create a list in Python?")

# Test Case 2: Python function question
code_focused_inference("Write a function to calculate factorial")

# Test Case 3: Non-coding question
code_focused_inference("What is the capital of France?")

# Test Case 4: Non-coding question
code_focused_inference("Tell me a joke")

# Test Case 5: Python error handling
code_focused_inference("How to fix IndentationError in Python?")

# Test Case 6: General question
code_focused_inference("What's the weather like today?")

print("\n" + "="*70)
print("TESTING COMPLETE")
print("="*70)

Torch version: 2.9.1+cu128
Transformers version: 4.36.0

Loading GPT-2 model and tokenizer...
✓ Model and tokenizer loaded successfully!
Model: gpt2
Number of parameters: 124,439,808


TESTING CODE-FOCUSED INFERENCE SYSTEM

User Prompt: How do I create a list in Python?
✓ This is a Python coding question. Generating response...

Response:
Python programming question: How do I create a list in Python?

Answer:
 (It's not so much that you can't do it, it's that it takes time and patience. You could do everything for free, but you need to be able to find and use it.)
… and we all know this because it was originally written by David Lippman.
.

User Prompt: Write a function to calculate factorial
✓ This is a Python coding question. Generating response...

Response:
Python programming question: Write a function to calculate factorial

Answer: write a program that will calculate the factoring function.
/\x00f\
:\0: \x01\:
.\d
\t\f
+\r
[\xa00\] = \xe00
(x)
In this case, the program will compu

In [8]:

def interactive_mode():
    """
    Interactive mode for testing the code-focused inference system.
    Users can ask questions in real-time.
    """
    print("\n" + "="*70)
    print("🐍 CODE-FOCUSED INFERENCE SYSTEM - INTERACTIVE MODE 🐍")
    print("="*70)
    print("Ask me Python coding questions!")
    print("Commands:")
    print("  - Type your question to get an answer")
    print("  - Type 'quit', 'exit', or 'q' to exit")
    print("  - Type 'help' for examples")
    print("="*70 + "\n")

    while True:
        # Get user input
        user_input = input("💬 Your question: ").strip()

        # Check for exit commands
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("\n👋 Exiting interactive mode. Goodbye!")
            break

        # Check for empty input
        if not user_input:
            print("⚠️  Please enter a question.\n")
            continue

        # Check for help command
        if user_input.lower() == 'help':
            print("\n📚 Example Python coding questions:")
            print("  - How do I create a dictionary in Python?")
            print("  - Write a function to find the maximum value in a list")
            print("  - How to handle file operations in Python?")
            print("  - What is list comprehension?")
            print("  - How to debug IndexError in Python?\n")
            continue

        # Process the question
        code_focused_inference(user_input)
        print()

# Run interactive mode
interactive_mode()


🐍 CODE-FOCUSED INFERENCE SYSTEM - INTERACTIVE MODE 🐍
Ask me Python coding questions!
Commands:
  - Type your question to get an answer
  - Type 'quit', 'exit', or 'q' to exit
  - Type 'help' for examples

💬 Your question: what is your name

User Prompt: what is your name
✗ This is NOT a Python coding question.

Response:
I'm sorry, but I can only answer questions related to Python coding. Please ask me a Python programming question!

💬 Your question: what is lists

User Prompt: what is lists
✓ This is a Python coding question. Generating response...

Response:
Python programming question: what is lists

Answer: lists are data structures that are often used by a program. A list is a list of values.
. One can see that if there is no value, then the program will use any of its arguments and returns the next value. This is called a data structure. List definitions are called "lines".
, a single line of code. The list's end is defined by an interface. You can get more information about the